In [1]:

import importlib
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
import functions
from model.cnn_model import Simple3DCNN, Simple3DCNNLarge, ProfileMLP


importlib.reload(functions)
from notebooks.OOF.functions import train_profile_mlp, predict_profile_mlp, train_cnn_model, predict_cnn_model


import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path


In [2]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1363
Liczba etykiet: 1362


In [3]:
data_processed_volume = np.load(
    project_dir / "data" / "processed_volumes_with_rotation_by_similarity.npz",
    allow_pickle=False,
)

X_volumes = data_processed_volume["X"].astype(np.float32)
y_volumes = data_processed_volume["y"].astype(np.int64)
uids_processed_volumes = data_processed_volume["uid"]

data_profiles = np.load(
    project_dir / "data" / "processed_profiles.npz",
    allow_pickle=False,
)

print(data_profiles.files)

uids_profiles = data_profiles["uid"]
y_profiles = data_profiles["is_pathologic"].astype(np.int64)

x_profile = data_profiles["x_profile"]
y_profile = data_profiles["y_profile"]
z_profile = data_profiles["z_profile"]
rotation_x_profile = data_profiles["rotation_x_profile"]
tilt_x_profiles = data_profiles["tilt_x_profiles"]
pitch_y_profiles = data_profiles["pitch_y_profiles"]

print("Volumes:", X_volumes.shape, y_volumes.shape, uids_processed_volumes.shape)
print("Profiles:", y_profiles.shape, uids_profiles.shape)

print("UID order equal:", np.array_equal(uids_processed_volumes, uids_profiles))
print("Labels equal:", np.array_equal(y_volumes, y_profiles))

assert np.array_equal(uids_processed_volumes, uids_profiles)
assert np.array_equal(y_volumes, y_profiles)

uid_to_idx = {
    uid: idx
    for idx, uid in enumerate(uids_processed_volumes)
}

['uid', 'is_pathologic', 'x_profile', 'y_profile', 'z_profile', 'rotation_x_profile', 'tilt_x_profiles', 'pitch_y_profiles']
Volumes: (1362, 1, 50, 60, 30) (1362,) (1362,)
Profiles: (1362,) (1362,)
UID order equal: True
Labels equal: True


In [4]:
split_df = pd.DataFrame({
    "uid": uids_processed_volumes,
    "is_pathologic": y_volumes,
})

train_main_df, temp_df = train_test_split(
    split_df,
    test_size=0.3,
    stratify=split_df["is_pathologic"],
    random_state=42,
)

valid_meta_df, test_final_df = train_test_split(
    temp_df,
    test_size=1/3,
    stratify=temp_df["is_pathologic"],
    random_state=42,
)

train_main_df = train_main_df.reset_index(drop=True)
valid_meta_df = valid_meta_df.reset_index(drop=True)
test_final_df = test_final_df.reset_index(drop=True)

train_main_df["split"] = "train_main"
valid_meta_df["split"] = "valid_meta"
test_final_df["split"] = "test_final"

train_main_df["fold"] = -1
valid_meta_df["fold"] = -1
test_final_df["fold"] = -1

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (_, val_idx) in enumerate(
    skf.split(
        train_main_df["uid"],
        train_main_df["is_pathologic"],
    )
):
    train_main_df.loc[val_idx, "fold"] = fold

oof_split_df = pd.concat(
    [
        train_main_df,
        valid_meta_df,
        test_final_df,
    ],
    ignore_index=True,
)

oof_split_df.to_csv(
    "oof_split.csv",
    index=False,
)

print(oof_split_df["split"].value_counts())
print(pd.crosstab(oof_split_df["split"], oof_split_df["is_pathologic"]))
print(pd.crosstab(train_main_df["fold"], train_main_df["is_pathologic"]))

assert oof_split_df["uid"].is_unique
assert set(train_main_df["fold"]) == set(range(5))

split
train_main    953
valid_meta    272
test_final    137
Name: count, dtype: int64
is_pathologic    0    1
split                  
test_final      62   75
train_main     430  523
valid_meta     123  149
is_pathologic   0    1
fold                  
0              86  105
1              86  105
2              86  105
3              86  104
4              86  104


In [5]:
print(oof_split_df.groupby("split")["is_pathologic"].mean())
print(train_main_df.groupby("fold")["is_pathologic"].mean())

split
test_final    0.547445
train_main    0.548793
valid_meta    0.547794
Name: is_pathologic, dtype: float64
fold
0    0.549738
1    0.549738
2    0.549738
3    0.547368
4    0.547368
Name: is_pathologic, dtype: float64


In [6]:
X_profiles = np.concatenate(
    [
        x_profile,
        y_profile,
        z_profile,
        rotation_x_profile,
    ],
    axis=1,
).astype(np.float32)
def add_result(results, model_name, fold, y_true, y_proba):
    results.append({
        "model": model_name,
        "fold": fold,
        "log_loss": log_loss(y_true, y_proba),
        "auroc": roc_auc_score(y_true, y_proba),
    })



In [7]:

results = []
for fold in range(5):
    print(f"\nFold {fold}")

    train_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] != fold),
        "uid"
    ].to_numpy()

    valid_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] == fold),
        "uid"
    ].to_numpy()

    train_idx = np.array([uid_to_idx[uid] for uid in train_uids])
    valid_idx = np.array([uid_to_idx[uid] for uid in valid_uids])

    y_train = y_volumes[train_idx]
    y_valid = y_volumes[valid_idx]

    print("train:", len(train_idx), "valid:", len(valid_idx))

    profile_logreg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=10000,
        random_state=42,
        solver="saga",
    )
    )

    param_grid = {
    "logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
    "logisticregression__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
}

    profile_logreg_grid = GridSearchCV(
        estimator=profile_logreg_pipeline,
        param_grid=param_grid,
        scoring="neg_log_loss",
        cv=3,
        n_jobs=-1,
    )

    profile_logreg_grid.fit(
        X_profiles[train_idx],
        y_train,
    )

    best_profile_logreg = profile_logreg_grid.best_estimator_

    joblib.dump(
        best_profile_logreg,
        f"models/profile_logreg_fold_{fold}.joblib",
    )

    p_profile_logreg = best_profile_logreg.predict_proba(
        X_profiles[valid_idx]
    )[:, 1]

    add_result(
        results=results,
        model_name="profile_logreg",
        fold=fold,
        y_true=y_valid,
        y_proba=p_profile_logreg,
    )

results



Fold 0
train: 762 valid: 191

Fold 1
train: 762 valid: 191

Fold 2
train: 762 valid: 191

Fold 3
train: 763 valid: 190

Fold 4
train: 763 valid: 190


[{'model': 'profile_logreg',
  'fold': 0,
  'log_loss': 0.41439083218574524,
  'auroc': 0.8942414174972314},
 {'model': 'profile_logreg',
  'fold': 1,
  'log_loss': 0.41586676239967346,
  'auroc': 0.8880398671096346},
 {'model': 'profile_logreg',
  'fold': 2,
  'log_loss': 0.4103754162788391,
  'auroc': 0.8955703211517165},
 {'model': 'profile_logreg',
  'fold': 3,
  'log_loss': 0.5313894152641296,
  'auroc': 0.8339669051878354},
 {'model': 'profile_logreg',
  'fold': 4,
  'log_loss': 0.4981141686439514,
  'auroc': 0.8353085867620752}]

In [8]:
results_df = pd.DataFrame(results)

summary_df = results_df.groupby("model").agg(
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
    auroc_mean=("auroc", "mean"),
    auroc_std=("auroc", "std"),
)

display(results_df)
display(summary_df)

results_df.to_csv(
    "results/profile_logreg_oof_results.csv",
    index=False,
)

,model,fold,log_loss,auroc
0,profile_logreg,0,0.414391,0.894241
1,profile_logreg,1,0.415867,0.888040
2,profile_logreg,2,0.410375,0.895570
3,profile_logreg,3,0.531389,0.833967
4,profile_logreg,4,0.498114,0.835309


,log_loss_mean,log_loss_std,auroc_mean,auroc_std
model,,,,
profile_logreg,0.454027,0.056704,0.869425,0.031887


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [8]:
def add_result_mlp(
    mlp_results,
    model_name,
    fold,
    y_true,
    y_proba,
    params=None,
    model_path=None,
):
    row = {
        "model": model_name,
        "fold": fold,
        "log_loss": log_loss(y_true, y_proba),
        "auroc": roc_auc_score(y_true, y_proba),
    }

    if params is not None:
        row["params"] = str(params)

    if model_path is not None:
        row["model_path"] = str(model_path)

    mlp_results.append(row)

In [9]:
mlp_results = []


for fold in range(5):
    print(f"\nFold {fold}")

    train_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] != fold),
        "uid"
    ].to_numpy()

    valid_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] == fold),
        "uid"
    ].to_numpy()

    train_idx = np.array([uid_to_idx[uid] for uid in train_uids])
    valid_idx = np.array([uid_to_idx[uid] for uid in valid_uids])

    y_train = y_volumes[train_idx]
    y_valid = y_volumes[valid_idx]

    print("train:", len(train_idx), "valid:", len(valid_idx))

    model_profile_mlp, best_valid_loss = train_profile_mlp(
            X_train=X_profiles[train_idx],
            y_train=y_train,
            X_valid=X_profiles[valid_idx],
            y_valid=y_valid,
            input_size=X_profiles.shape[1],
            epochs=300,
            batch_size=64,
            learning_rate=1e-4,
            weight_decay=1e-4,
            patience=30,
        )

    model_path = f"models/profile_mlp_fold_{fold}.pth"

    torch.save(
            model_profile_mlp.state_dict(),
            model_path,
    )

    p_profile_mlp = predict_profile_mlp(
            model_profile_mlp,
            X_profiles[valid_idx],
    )

    add_result_mlp(
            mlp_results=mlp_results,
            model_name="profile_mlp",
            fold=fold,
            y_true=y_valid,
            y_proba=p_profile_mlp,
            params={
                "best_valid_loss": best_valid_loss,
                "model_path": str(model_path),
            },
        )

    print(mlp_results[-1])




Fold 0
train: 762 valid: 191
{'model': 'profile_mlp', 'fold': 0, 'log_loss': 0.43725958466529846, 'auroc': 0.8822812846068661, 'params': "{'best_valid_loss': 0.43725961446762085, 'model_path': 'models/profile_mlp_fold_0.pth'}"}

Fold 1
train: 762 valid: 191
{'model': 'profile_mlp', 'fold': 1, 'log_loss': 0.3943496644496918, 'auroc': 0.8987818383167221, 'params': "{'best_valid_loss': 0.3943496644496918, 'model_path': 'models/profile_mlp_fold_1.pth'}"}

Fold 2
train: 762 valid: 191
{'model': 'profile_mlp', 'fold': 2, 'log_loss': 0.4082057476043701, 'auroc': 0.8900332225913622, 'params': "{'best_valid_loss': 0.4082058072090149, 'model_path': 'models/profile_mlp_fold_2.pth'}"}

Fold 3
train: 763 valid: 190
{'model': 'profile_mlp', 'fold': 3, 'log_loss': 0.4706745445728302, 'auroc': 0.8528622540250448, 'params': "{'best_valid_loss': 0.470674604177475, 'model_path': 'models/profile_mlp_fold_3.pth'}"}

Fold 4
train: 763 valid: 190
{'model': 'profile_mlp', 'fold': 4, 'log_loss': 0.48935499787

In [10]:
results_df = pd.DataFrame(mlp_results)

summary_df = results_df.groupby("model").agg(
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
    auroc_mean=("auroc", "mean"),
    auroc_std=("auroc", "std"),
)

display(results_df)
display(summary_df)

results_df.to_csv(
    "results/profile_mlp_oof_results.csv",
    index=False,
)

,model,fold,log_loss,auroc,params
0,profile_mlp,0,0.437260,0.882281,"{'best_valid_loss': 0.43725961446762085, 'mode..."
1,profile_mlp,1,0.394350,0.898782,"{'best_valid_loss': 0.3943496644496918, 'model..."
2,profile_mlp,2,0.408206,0.890033,"{'best_valid_loss': 0.4082058072090149, 'model..."
3,profile_mlp,3,0.470675,0.852862,"{'best_valid_loss': 0.470674604177475, 'model_..."
4,profile_mlp,4,0.489355,0.842129,"{'best_valid_loss': 0.4893549978733063, 'model..."


,log_loss_mean,log_loss_std,auroc_mean,auroc_std
model,,,,
profile_mlp,0.439969,0.040247,0.873217,0.024491


In [14]:
results_small_cnn = []

for fold in range(5):
    print(f"\nFold {fold}")

    train_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] != fold),
        "uid"
    ].to_numpy()

    valid_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] == fold),
        "uid"
    ].to_numpy()

    train_idx = np.array([uid_to_idx[uid] for uid in train_uids])
    valid_idx = np.array([uid_to_idx[uid] for uid in valid_uids])

    y_train = y_volumes[train_idx]
    y_valid = y_volumes[valid_idx]

    print("train:", len(train_idx), "valid:", len(valid_idx))

    # SMALL CNN
    small_model = Simple3DCNN()
    small_path = f"models/small_cnn_fold_{fold}.pth"

    small_model, small_valid_proba, small_valid_loss = train_cnn_model(
        model=small_model,
        X_train=X_volumes[train_idx],
        y_train=y_volumes[train_idx],
        X_valid=X_volumes[valid_idx],
        y_valid=y_volumes[valid_idx],
        epochs=1500,
        batch_size=8,
        learning_rate=1e-4,
        weight_decay=1e-4,
        patience=150,
    )

    torch.save(small_model.state_dict(), small_path)

    add_result_mlp(
        mlp_results=results_small_cnn,
        model_name="small_cnn",
        fold=fold,
        y_true=y_valid,
        y_proba=small_valid_proba,
        params={
            "epochs": 1500,
            "batch_size": 8,
            "lr": 1e-4,
            "weight_decay": 1e-4,
            "patience": 150,
        },
        model_path=small_path,
    )



Fold 0
train: 762 valid: 191
Epoch 001/1500 | train_loss: 0.6903 | valid_loss: 0.6875 | best: 0.6875 | patience: 0/150
Epoch 002/1500 | train_loss: 0.6858 | valid_loss: 0.6852 | best: 0.6852 | patience: 0/150
Epoch 003/1500 | train_loss: 0.6847 | valid_loss: 0.6848 | best: 0.6848 | patience: 0/150
Epoch 004/1500 | train_loss: 0.6854 | valid_loss: 0.6847 | best: 0.6847 | patience: 0/150
Epoch 005/1500 | train_loss: 0.6881 | valid_loss: 0.6844 | best: 0.6844 | patience: 0/150
Epoch 006/1500 | train_loss: 0.6858 | valid_loss: 0.6840 | best: 0.6840 | patience: 0/150
Epoch 007/1500 | train_loss: 0.6840 | valid_loss: 0.6835 | best: 0.6835 | patience: 0/150
Epoch 008/1500 | train_loss: 0.6812 | valid_loss: 0.6825 | best: 0.6825 | patience: 0/150
Epoch 009/1500 | train_loss: 0.6834 | valid_loss: 0.6818 | best: 0.6818 | patience: 0/150
Epoch 010/1500 | train_loss: 0.6807 | valid_loss: 0.6810 | best: 0.6810 | patience: 0/150
Epoch 011/1500 | train_loss: 0.6842 | valid_loss: 0.6793 | best: 0.679

In [15]:
results_df = pd.DataFrame(results_small_cnn)

summary_df = results_df.groupby("model").agg(
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
    auroc_mean=("auroc", "mean"),
    auroc_std=("auroc", "std"),
)

display(results_df)
display(summary_df)

results_df.to_csv(
    "results/profile_smal_cnn_oof_results.csv",
    index=False,
)

,model,fold,log_loss,auroc,params,model_path
0,small_cnn,0,0.269521,0.955925,"{'epochs': 1500, 'batch_size': 8, 'lr': 0.0001...",models/small_cnn_fold_0.pth
1,small_cnn,1,0.327389,0.932226,"{'epochs': 1500, 'batch_size': 8, 'lr': 0.0001...",models/small_cnn_fold_1.pth
2,small_cnn,2,0.238415,0.964673,"{'epochs': 1500, 'batch_size': 8, 'lr': 0.0001...",models/small_cnn_fold_2.pth
3,small_cnn,3,0.404473,0.900939,"{'epochs': 1500, 'batch_size': 8, 'lr': 0.0001...",models/small_cnn_fold_3.pth
4,small_cnn,4,0.251899,0.961315,"{'epochs': 1500, 'batch_size': 8, 'lr': 0.0001...",models/small_cnn_fold_4.pth


,log_loss_mean,log_loss_std,auroc_mean,auroc_std
model,,,,
small_cnn,0.298339,0.068349,0.943016,0.026727


In [18]:
results_large_cnn = []

for fold in range(5):
    print(f"\nFold {fold}")

    train_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] != fold),
        "uid"
    ].to_numpy()

    valid_uids = oof_split_df.loc[
        (oof_split_df["split"] == "train_main")
        & (oof_split_df["fold"] == fold),
        "uid"
    ].to_numpy()

    train_idx = np.array([uid_to_idx[uid] for uid in train_uids])
    valid_idx = np.array([uid_to_idx[uid] for uid in valid_uids])

    y_train = y_volumes[train_idx]
    y_valid = y_volumes[valid_idx]

    print("train:", len(train_idx), "valid:", len(valid_idx))

    # Large CNN
    large_model = Simple3DCNNLarge()
    large_path = f"models/large_cnn_fold_{fold}.pth"

    large_model, large_valid_proba, large_valid_loss = train_cnn_model(
        model=large_model,
        X_train=X_volumes[train_idx],
        y_train=y_volumes[train_idx],
        X_valid=X_volumes[valid_idx],
        y_valid=y_volumes[valid_idx],
        epochs=2500,
        batch_size=8,
        learning_rate=1e-4,
        weight_decay=1e-4,
        patience=150,
    )

    torch.save(large_model.state_dict(), large_path)

    add_result_mlp(
        mlp_results=results_large_cnn,
        model_name="large_cnn",
        fold=fold,
        y_true=y_valid,
        y_proba=large_valid_proba,
        params={
            "epochs": 2500,
            "batch_size": 8,
            "lr": 1e-4,
            "weight_decay": 1e-4,
            "patience": 150,
        },
        model_path=large_path,
    )



Fold 0
train: 762 valid: 191
Epoch 001/2500 | train_loss: 0.6878 | valid_loss: 0.6879 | best: 0.6879 | patience: 0/150
Epoch 002/2500 | train_loss: 0.6879 | valid_loss: 0.6871 | best: 0.6871 | patience: 0/150
Epoch 003/2500 | train_loss: 0.6868 | valid_loss: 0.6867 | best: 0.6867 | patience: 0/150
Epoch 004/2500 | train_loss: 0.6859 | valid_loss: 0.6858 | best: 0.6858 | patience: 0/150
Epoch 005/2500 | train_loss: 0.6850 | valid_loss: 0.6849 | best: 0.6849 | patience: 0/150
Epoch 006/2500 | train_loss: 0.6859 | valid_loss: 0.6835 | best: 0.6835 | patience: 0/150
Epoch 007/2500 | train_loss: 0.6830 | valid_loss: 0.6810 | best: 0.6810 | patience: 0/150
Epoch 008/2500 | train_loss: 0.6799 | valid_loss: 0.6768 | best: 0.6768 | patience: 0/150
Epoch 009/2500 | train_loss: 0.6740 | valid_loss: 0.6729 | best: 0.6729 | patience: 0/150
Epoch 010/2500 | train_loss: 0.6720 | valid_loss: 0.6638 | best: 0.6638 | patience: 0/150
Epoch 011/2500 | train_loss: 0.6625 | valid_loss: 0.6513 | best: 0.651

In [19]:
results_df = pd.DataFrame(results_large_cnn)

summary_df = results_df.groupby("model").agg(
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
    auroc_mean=("auroc", "mean"),
    auroc_std=("auroc", "std"),
)

display(results_df)
display(summary_df)

results_df.to_csv(
    "results/profile_large_cnn_oof_results.csv",
    index=False,
)

,model,fold,log_loss,auroc,params,model_path
0,large_cnn,0,0.313108,0.939646,"{'epochs': 2500, 'batch_size': 8, 'lr': 0.0001...",models/large_cnn_fold_0.pth
1,large_cnn,1,0.311165,0.935659,"{'epochs': 2500, 'batch_size': 8, 'lr': 0.0001...",models/large_cnn_fold_1.pth
2,large_cnn,2,0.263313,0.952270,"{'epochs': 2500, 'batch_size': 8, 'lr': 0.0001...",models/large_cnn_fold_2.pth
3,large_cnn,3,0.421051,0.897697,"{'epochs': 2500, 'batch_size': 8, 'lr': 0.0001...",models/large_cnn_fold_3.pth
4,large_cnn,4,0.267280,0.956507,"{'epochs': 2500, 'batch_size': 8, 'lr': 0.0001...",models/large_cnn_fold_4.pth


,log_loss_mean,log_loss_std,auroc_mean,auroc_std
model,,,,
large_cnn,0.315183,0.063667,0.936356,0.023266


In [11]:
train_main_uids = oof_split_df.loc[oof_split_df["split"] == "train_main", "uid"].values
valid_meta_uids = oof_split_df.loc[oof_split_df["split"] == "valid_meta", "uid"].values
test_final_uids = oof_split_df.loc[oof_split_df["split"] == "test_final", "uid"].values

uid_to_idx = {uid: idx for idx, uid in enumerate(uids_processed_volumes)}

valid_meta_idx = np.array([uid_to_idx[uid] for uid in valid_meta_uids])
test_final_idx = np.array([uid_to_idx[uid] for uid in test_final_uids])

In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_small_cnn(fold):
    model = Simple3DCNN()
    path = f"models/small_cnn_fold_{fold}.pth"

    model.load_state_dict(
        torch.load(path, map_location=device)
    )

    model = model.to(device)
    model.eval()

    return model


def load_large_cnn(fold):
    model = Simple3DCNNLarge()
    path = f"models/large_cnn_fold_{fold}.pth"

    model.load_state_dict(
        torch.load(path, map_location=device)
    )

    model = model.to(device)
    model.eval()

    return model
def load_profile_mlp(fold, input_size):
    model = ProfileMLP(input_size=input_size)
    path = f"models/profile_mlp_fold_{fold}.pth"

    model.load_state_dict(
        torch.load(path, map_location=device)
    )

    model = model.to(device)
    model.eval()

    return model
def load_profile_logreg(fold):
    path = f"models/profile_logreg_fold_{fold}.joblib"
    model = joblib.load(path)

    return model

In [12]:
valid_preds = {
    "profile_logreg": [],
    "profile_mlp": [],
    "small_cnn": [],
    "large_cnn": [],
}

test_preds = {
    "profile_logreg": [],
    "profile_mlp": [],
    "small_cnn": [],
    "large_cnn": [],
}

for fold in range(5):
    print(f"Fold {fold}")

    profile_logreg_model = load_profile_logreg(fold)
    profile_mlp_model = load_profile_mlp(
        fold,
        input_size=X_profiles.shape[1],
    )
    small_cnn_model = load_small_cnn(fold)
    large_cnn_model = load_large_cnn(fold)

    valid_preds["profile_logreg"].append(
        profile_logreg_model.predict_proba(X_profiles[valid_meta_idx])[:, 1]
    )

    test_preds["profile_logreg"].append(
        profile_logreg_model.predict_proba(X_profiles[test_final_idx])[:, 1]
    )

    valid_preds["profile_mlp"].append(
        predict_profile_mlp(profile_mlp_model, X_profiles[valid_meta_idx])
    )

    test_preds["profile_mlp"].append(
        predict_profile_mlp(profile_mlp_model, X_profiles[test_final_idx])
    )

    valid_preds["small_cnn"].append(
        predict_cnn_model(small_cnn_model, X_volumes[valid_meta_idx])
    )

    test_preds["small_cnn"].append(
        predict_cnn_model(small_cnn_model, X_volumes[test_final_idx])
    )

    valid_preds["large_cnn"].append(
        predict_cnn_model(large_cnn_model, X_volumes[valid_meta_idx])
    )

    test_preds["large_cnn"].append(
        predict_cnn_model(large_cnn_model, X_volumes[test_final_idx])
    )

Fold 0


C:\Users\abram\AppData\Local\Temp\ipykernel_43452\2958847578.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device)
C:\Users\abram\AppDat

Fold 1


C:\Users\abram\AppData\Local\Temp\ipykernel_43452\2958847578.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device)
C:\Users\abram\AppDat

Fold 2


C:\Users\abram\AppData\Local\Temp\ipykernel_43452\2958847578.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device)
C:\Users\abram\AppDat

Fold 3


C:\Users\abram\AppData\Local\Temp\ipykernel_43452\2958847578.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device)
C:\Users\abram\AppDat

Fold 4


C:\Users\abram\AppData\Local\Temp\ipykernel_43452\2958847578.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device)
C:\Users\abram\AppDat

In [15]:
X_meta_valid_mean = np.column_stack([
    np.mean(valid_preds["profile_logreg"], axis=0),
    np.mean(valid_preds["profile_mlp"], axis=0),
    np.mean(valid_preds["small_cnn"], axis=0),
    np.mean(valid_preds["large_cnn"], axis=0),
])

X_meta_test_mean = np.column_stack([
    np.mean(test_preds["profile_logreg"], axis=0),
    np.mean(test_preds["profile_mlp"], axis=0),
    np.mean(test_preds["small_cnn"], axis=0),
    np.mean(test_preds["large_cnn"], axis=0),
])

In [16]:
X_meta_valid_all = np.column_stack([
    *valid_preds["profile_logreg"],
    *valid_preds["profile_mlp"],
    *valid_preds["small_cnn"],
    *valid_preds["large_cnn"],
])

X_meta_test_all = np.column_stack([
    *test_preds["profile_logreg"],
    *test_preds["profile_mlp"],
    *test_preds["small_cnn"],
    *test_preds["large_cnn"],
])

In [17]:
print(X_meta_valid_mean.shape)
print(X_meta_test_mean.shape)

print(X_meta_valid_all.shape)
print(X_meta_test_all.shape)

(272, 4)
(137, 4)
(272, 20)
(137, 20)


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
import pandas as pd

y_meta_valid = y_volumes[valid_meta_idx]
y_meta_test = y_volumes[test_final_idx]

meta_results = []

for name, Xv, Xt in [
    ("mean_folds", X_meta_valid_mean, X_meta_test_mean),
    ("all_folds", X_meta_valid_all, X_meta_test_all),
]:
    meta_model = LogisticRegression(
        C=0.1,
        max_iter=1000,
        solver="lbfgs",
    )

    meta_model.fit(Xv, y_meta_valid)

    valid_proba = meta_model.predict_proba(Xv)[:, 1]
    test_proba = meta_model.predict_proba(Xt)[:, 1]

    meta_results.append({
        "model": name,
        "valid_log_loss": log_loss(y_meta_valid, valid_proba),
        "valid_auroc": roc_auc_score(y_meta_valid, valid_proba),
        "test_log_loss": log_loss(y_meta_test, test_proba),
        "test_auroc": roc_auc_score(y_meta_test, test_proba),
    })

meta_results_df = pd.DataFrame(meta_results)
meta_results_df

,model,valid_log_loss,valid_auroc,test_log_loss,test_auroc
0,mean_folds,0.402446,0.923665,0.374677,0.938925
1,all_folds,0.347939,0.932940,0.316990,0.940860


In [19]:
blend_valid = X_meta_valid_mean.mean(axis=1)
blend_test = X_meta_test_mean.mean(axis=1)

blend_results = {
    "model": "simple_blend_mean_4_models",
    "valid_log_loss": log_loss(y_meta_valid, blend_valid),
    "valid_auroc": roc_auc_score(y_meta_valid, blend_valid),
    "test_log_loss": log_loss(y_meta_test, blend_test),
    "test_auroc": roc_auc_score(y_meta_test, blend_test),
}

pd.concat([
    meta_results_df,
    pd.DataFrame([blend_results]),
], ignore_index=True)

,model,valid_log_loss,valid_auroc,test_log_loss,test_auroc
0,mean_folds,0.402446,0.923665,0.374677,0.938925
1,all_folds,0.347939,0.932940,0.316990,0.940860
2,simple_blend_mean_4_models,0.374393,0.912315,0.340804,0.931613


In [20]:
joblib.dump(
    meta_model,
    "models/meta_logreg_all_folds.joblib",
)

['models/meta_logreg_all_folds.joblib']

In [21]:
final_predictions_df = pd.DataFrame({
    "uid": test_final_uids,
    "y_true": y_meta_test,
    "probability": test_proba,
})

final_predictions_df.to_csv(
    "meta_test_predictions.csv",
    index=False,
)

## Podsumowanie procesu OOF i stackingu

W eksperymencie zastosowano podział danych na trzy niezależne części: `train_main`, `valid_meta` oraz `test_final`. Zbiór `train_main` został dodatkowo podzielony na pięć foldów za pomocą `StratifiedKFold`, dzięki czemu zachowano zbliżone proporcje klas w każdym foldzie.

Dla każdego folda wytrenowano cztery modele bazowe:

- regresję logistyczną wykorzystującą profile obrazów,
- sieć MLP wykorzystującą profile obrazów,
- mniejszy model 3D CNN wykorzystujący pełne wolumeny,
- większy model 3D CNN wykorzystujący pełne wolumeny.

Łącznie powstało 20 modeli bazowych: cztery rodzaje modeli wytrenowane osobno dla pięciu foldów. Po zakończeniu ich treningu modele nie były już ponownie dopasowywane.

Sprawdzono trzy sposoby połączenia modeli:

1. `mean_folds` – uśrednienie predykcji pięciu foldów osobno dla każdego rodzaju modelu, a następnie przekazanie czterech cech do meta-modelu;
2. `all_folds` – przekazanie do meta-modelu wszystkich 20 predykcji bez wcześniejszego uśredniania;
3. `simple_blend_mean_4_models` – proste uśrednienie predykcji czterech modeli bez trenowania meta-modelu.

Rolę meta-modelu pełniła regresja logistyczna trenowana na zbiorze `valid_meta`. Ostateczna ocena została przeprowadzona na niezależnym zbiorze `test_final`, który nie uczestniczył ani w treningu modeli bazowych, ani w treningu meta-modelu.

Najlepszy wynik uzyskał wariant `all_folds`:

- **test log loss: 0.3170**
- **test AUROC: 0.9409**

Zastosowany proces ogranicza ryzyko data leakage poprzez rozdzielenie danych używanych do treningu modeli bazowych, treningu meta-modelu oraz końcowego testu. Technicznie meta-model został wytrenowany na osobnym zbiorze `valid_meta`, a nie na klasycznych predykcjach OOF ze zbioru `train_main`. Jest to więc połączenie treningu foldowego modeli bazowych, ensemble oraz stackingu z wydzielonym zbiorem dla meta-modelu.